# Projeto: Funções de Tratamento de Dados Meteorológicos – Clima Tempo

## Descrição Geral
Este notebook contém a definição das funções responsáveis pela limpeza, normalização e padronização de colunas meteorológicas provenientes da fonte Clima Tempo. As funções foram desenvolvidas para serem utilizadas em pipelines de preparação de dados, visando a consistência e qualidade dos dados a serem persistidos na camada silver.

## Objetivos

- Criar funções reutilizáveis para tratamento de colunas meteorológicas específicas.
- Garantir a padronização de formatos e unidades (ex: °C, %, km/h, mph).
- Eliminar ou substituir valores nulos, vazios ou inválidos.
- Preparar os dados para ingestão em camadas analíticas posteriores.


## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Databricks Notebooks
- Delta Lake

---

## Funções Definidas

| Função                     | Descrição                                                                 |
|---------------------------|---------------------------------------------------------------------------|
| `mostrar_duplicatas`            | Lista linhas duplicadas no dataframe conforme parâmetro indicado |
| `remover_duplicatas`            | Realiza a remoção de duplicados conforme o parâmetro indicado |
| `alterar_tipos`            | Transforma o tipo do dado para date, timestamp ou decimal |
| `renomear_colunas`            | Atualiza o nome das colunas conforme mapeamento criado em dicionário |
| `nomalizar_nomes_colunas`            | Atualiza o nome das colunas removendo caracteres especiais e incluindo "_" entre os espaços |
| `tratar_celulas`          | Aplica, em sequência, todas as funções de tratamento definidas no notebook. |
| `tratar_precipitacao`     | Separa o valor da precipitação e a probabilidade de ocorrência (%) em colunas distintas. |
| `limpar_simbolos`         | Remove os símbolos "°" e "%" das colunas de temperatura e umidade.         |
| `tratar_temperaturas_umidades` | Valida e substitui valores inconsistentes de temperatura, umidade e precipitação. |
| `tratar_vento`            | Converte valores de velocidade do vento para tipo numérico, padronizando unidades e formatos. |
| `normalizar_nome_municipio`            | Transforma nomes das capitais incorretos (sem acentos), para os nomes corretos |


## Colunas Tratadas da função tratar_celulas()

- `precipitacao`: separada em `precipitacao` (valor em mm) e `prob_precipitacao` (%)
- `temp_max`, `temp_min`: símbolos removidos e valores inválidos substituídos por 9999
- `umid_max`, `umid_min`: símbolos removidos e valores inválidos substituídos por -1
- `vel_vento`: convertida para double em km/h, com tratamento de valores em "mph" e campos vazios

---

### Histórico de Atualizações

| Data       | Responsável  | Versão | Descrição                                      |
|------------|--------------|--------|-----------------------------------------------|
| 16/05/2025 | Lucas Bringmann  | 1.0    | Criação inicial das funções de tratamento     |
| 16/05/2025 | Evelise Dias  | 1.1    | Refatoração da função trata_celulas em funções menores para melhor legibilidade do código    |
| 16/05/2025 | Lucas Berta  | 1.2    | Refatoração de funções e criação de docstrings e typehints     |
| 18/05/2025 | Lucas Bringmann  | 1.3    | Adição de função para normalização de nomes das capitais     |
| 19/05/2025 | Evelise Dias  | 1.4   | Atualização de documentação    |





In [0]:
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.functions import lit, col, explode, regexp, regexp_replace, when, regexp_extract, to_timestamp, to_date
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, DateType, DecimalType
from delta.tables import DeltaTable

In [0]:
def remover_duplicatas(dataframe, campos):
    """
    Remove duplicatas de um DataFrame com base em colunas específicas e informa a quantidade removida.

    Args:
        dataframe (DataFrame): DataFrame original.
        campos (list): Lista de colunas para considerar ao remover duplicatas.

    Returns:
        DataFrame: DataFrame sem registros duplicados.
    """
    df_dups = dataframe.groupBy(campos).count().filter("count > 1")
    print("numero de duplicatas " + f"{df_dups.count()}")
    return dataframe.dropDuplicates(campos)

In [0]:
def alterar_tipos(df: DataFrame, colunas: list, tipo: str, data_formato: str="yyyy-MM-dd", precisao: int = 5, escala: int = 2) -> DataFrame:
    """
    Altera o tipo de dados de colunas específicas em um DataFrame.

    Args:
        df (DataFrame): DataFrame original.
        colunas (list): Lista de colunas a serem convertidas.
        tipo (str): Tipo de dado desejado (ex: 'int', 'double', 'string', 'date').
        data_formato (str): Formato da data caso isDate=True. Padrão: 'yyyy-MM-dd'.

    Returns:
        DataFrame: DataFrame com colunas convertidas.
    """
    for coluna in colunas:
        if tipo == "date":
            df = df.withColumn(coluna, to_date(col(coluna), data_formato))
        elif tipo == "timestamp":
            df = df.withColumn(coluna, to_timestamp(col(coluna), data_formato))
        elif tipo == "decimal":
            df = df.withColumn(coluna, col(coluna).cast(f"decimal({precisao}, {escala})"))
        else:
            df = df.withColumn(coluna, col(coluna).cast(tipo))
    return df

In [0]:
def renomear_colunas(df: DataFrame, mapeamento: dict) -> DataFrame:
    """
    Renomeia colunas de um DataFrame com base em um dicionário de mapeamento.

    Esta função percorre um dicionário onde as chaves representam os nomes antigos
    das colunas e os valores representam os novos nomes. Se uma coluna existente no
    DataFrame corresponder a uma chave no dicionário, ela será renomeada para o novo
    nome. Caso contrário, um aviso será registrado informando que a coluna não foi encontrada.

    Args:
        df (pyspark.sql.DataFrame): DataFrame original contendo as colunas a serem renomeadas.
        mapeamento (dict): Dicionário onde as chaves são os nomes originais das colunas e os valores
                           são os novos nomes desejados.

    Returns:
        pyspark.sql.DataFrame: DataFrame com as colunas renomeadas.
    """
    for nome_antigo, nome_novo in mapeamento.items():
        if nome_antigo in df.columns:
            logging.info(f"Renomeando coluna '{nome_antigo}' para '{nome_novo}'")
            df = df.withColumnRenamed(nome_antigo, nome_novo)
        else: 
            logging.warning(f"Coluna '{nome_antigo}' não encontrada no DataFrame.")
    return df

In [0]:
def normalizar_nomes_colunas(df: DataFrame) -> DataFrame:
    """
    Normaliza os nomes das colunas do DataFrame, convertendo caracteres especiais,
    substituindo espaços por underscore e removendo caracteres não alfanuméricos.

    Args:
        df (DataFrame): DataFrame original.

    Returns:
        DataFrame: DataFrame com nomes de colunas normalizados.
    """
    for column in df.columns:
        new_column = column.lower().replace(' ', '_')  # Converte caracteres
        new_column = ''.join(c if c.isalnum() or c == '_' else '' for c in new_column)  # Remove caracteres não alfanuméricos
        if new_column and new_column[0].isdigit():
            new_column = f"col_{new_column}"  # Ajusta colunas que começam com números

        # Correção: `translate` aplicado diretamente no DataFrame
        df = df.withColumnRenamed(column, new_column)

    return df

In [0]:
def tratar_celulas(df: DataFrame) -> DataFrame:
    """
    Aplica uma série de tratamentos e limpezas em um DataFrame meteorológico.

    Esta função encadeia múltiplas etapas de pré-processamento nos dados, incluindo
    o tratamento de valores de precipitação, limpeza de símbolos indesejados,
    ajustes em colunas de temperatura e umidade, e padronização de dados relacionados ao vento.

    Args:
        df (DataFrame): DataFrame contendo os dados meteorológicos brutos.

    Returns:
        DataFrame: DataFrame processado e limpo, pronto para análise.
    """
    df = tratar_precipitacao(df)
    df = limpar_simbolos(df)
    df = tratar_temperaturas_umidades(df)
    df = tratar_vento(df)
    return df

In [0]:
def tratar_precipitacao(df: DataFrame) -> DataFrame:
    """
    Trata os dados da coluna de precipitação em um DataFrame meteorológico.

    Extrai e converte informações da coluna 'precipitacao', separando os valores
    de probabilidade de precipitação (quando presentes em porcentagem) e os valores
    de volume de precipitação (em milímetros). Cria uma nova coluna chamada 
    'prob_precipitacao' com a probabilidade como valor numérico, e transforma a 
    coluna original 'precipitacao' em valores numéricos representando milímetros.

    Args:
        df (DataFrame): DataFrame contendo a coluna 'precipitacao' com valores em 
                        texto, podendo incluir porcentagens (ex: "30%") e milímetros 
                        (ex: "12mm").

    Returns:
        DataFrame: DataFrame com a coluna 'precipitacao' tratada como valor numérico
                   e a nova coluna 'prob_precipitacao' adicionada.
    """
    return df.withColumn(
        "prob_precipitacao",
        when(
            col("precipitacao").contains("%"),
            regexp_extract(col("precipitacao"), r"(\d+(?:\.\d+)?)%", 1).cast("double")
        ).when(
            col("precipitacao").rlike("^\d+(\.\d+)?%$"),
            regexp_extract(col("precipitacao"), r"(\d+(?:\.\d+)?)%", 1).cast("double")
        ).otherwise(0)
    ).withColumn(
        "precipitacao",
        when(
            col("precipitacao").contains("mm"),
            regexp_extract(col("precipitacao"), r"(\d+\.?\d*)", 1).cast("double")
        ).when(
            col("precipitacao").rlike(r"^\d+\.?\d*$"),
            col("precipitacao").cast("double")
        ).otherwise(0)
    )

In [0]:
def limpar_simbolos(df: DataFrame) -> DataFrame:
    """
    Remove símbolos indesejados das colunas de temperatura e umidade em um DataFrame meteorológico.

    Remove o símbolo de grau ("°") das colunas de temperatura máxima e mínima, 
    e o símbolo de porcentagem ("%") das colunas de umidade máxima e mínima, 
    caso essas colunas estejam presentes no DataFrame.

    Args:
        df (DataFrame): DataFrame contendo colunas opcionais como 'temp_max', 
                        'temp_min', 'umid_max' e 'umid_min', com valores que 
                        podem incluir os símbolos "°" ou "%".

    Returns:
        DataFrame: DataFrame com os símbolos removidos das colunas correspondentes.
    """
    if "temp_max" in df.columns:
        df = df.withColumn("temp_max", regexp_replace(col("temp_max"), "°", ""))
    if "temp_min" in df.columns:
        df = df.withColumn("temp_min", regexp_replace(col("temp_min"), "°", ""))
    if "umid_max" in df.columns:
        df = df.withColumn("umid_max", regexp_replace(col("umid_max"), "%", ""))
    if "umid_min" in df.columns:
        df = df.withColumn("umid_min", regexp_replace(col("umid_min"), "%", ""))
    return df

In [0]:
def tratar_temperaturas_umidades(df: DataFrame) -> DataFrame:
    """
    Realiza o tratamento de dados nas colunas de temperatura, umidade e precipitação.

    Verifica se os valores nas colunas de temperatura máxima e mínima ('temp_max', 'temp_min') 
    e de umidade máxima e mínima ('umid_max', 'umid_min') são válidos segundo seus formatos esperados.

    Também trata as colunas 'precipitacao' e 'prob_precipitacao', convertendo os valores para 
    tipo numérico.

    Args:
        df (DataFrame): DataFrame contendo colunas de dados meteorológicos.

    Returns:
        DataFrame: DataFrame com os dados tratados e padronizados.
    """
    if "temp_max" in df.columns:
        df = df.withColumn(
            "temp_max",
            when(col("temp_max").rlike(r"^\d+(\.\d+)?$"), col("temp_max")).otherwise(None)
        )
    if "temp_min" in df.columns:
        df = df.withColumn(
            "temp_min",
            when(col("temp_min").rlike(r"^\d+(\.\d+)?$"), col("temp_min")).otherwise(None)
        )
    if "umid_max" in df.columns:
        df = df.withColumn(
            "umid_max",
            when(col("umid_max").rlike(r"^\d+$"), col("umid_max")).otherwise(None)
        )
    if "umid_min" in df.columns:
        df = df.withColumn(
            "umid_min",
            when(col("umid_min").rlike(r"^\d+$"), col("umid_min")).otherwise(None)
        )
    
    if "precipitacao" in df.columns:
        df = df.withColumn(
            "precipitacao",
            when(
                col("precipitacao").cast("double").isNull() | (col("precipitacao").cast("double") == 0),
                lit(None)
            ).otherwise(col("precipitacao").cast("double"))
        )
    if "prob_precipitacao" in df.columns:
        df = df.withColumn(
            "prob_precipitacao",
            when(
                col("prob_precipitacao").cast("double").isNull() | (col("prob_precipitacao").cast("double") == 0),
                lit(None)
            ).otherwise(col("prob_precipitacao").cast("double"))
        )
    return df

In [0]:
def tratar_vento(df: DataFrame) -> DataFrame:
    """
    Trata e padroniza os dados da coluna de velocidade do vento em um DataFrame meteorológico.

    Converte os valores da coluna 'vel_vento' para tipo numérico (double), tratando diferentes
    unidades de medida como "km/h" e "mph", além de normalizar o uso de vírgulas e pontos
    em números decimais.

    - Valores em "km/h" são extraídos e convertidos diretamente.
    - Valores em "mph" são convertidos para km/h utilizando o fator 1.60934.
    - Valores sem unidade explícita, mas numéricos, são convertidos diretamente.

    Args:
        df (DataFrame): DataFrame contendo a coluna 'vel_vento' com valores de velocidade do vento
                        possivelmente em diferentes formatos e unidades.

    Returns:
        DataFrame: DataFrame com a coluna 'vel_vento' tratada e convertida para tipo double (em km/h).
    """
    if "vel_vento" not in df.columns:
        return df

    return df.withColumn(
        "vel_vento",
        when(
            col("vel_vento").isNull() | (col("vel_vento") == ""),
            lit(-1.0)
        ).when(
            col("vel_vento").contains("km/h") & col("vel_vento").rlike(r"(\d+([.,]\d+)?)"),
            regexp_replace(
                regexp_extract(col("vel_vento"), r"(\d+([.,]\d+)?)", 1),
                ",",
                "."
            ).cast("double")
        ).when(
            col("vel_vento").contains("mph") & col("vel_vento").rlike(r"(\d+([.,]\d+)?)"),
            (
                regexp_replace(
                    regexp_extract(col("vel_vento"), r"(\d+([.,]\d+)?)", 1),
                    ",",
                    "."
                ).cast("double") * lit(1.60934)
            )
        ).otherwise(
            regexp_replace(col("vel_vento"), ",", ".").cast("double")
        )
    )


In [0]:
from pyspark.sql.functions import when, col, trim, lower

def normalizar_nome_municipio(df, nome_correcoes, coluna):
    """
    Normaliza a coluna 'nom_municipio' do DataFrame usando o dicionário de correções.
    
    Args:
        df: DataFrame com a coluna 'nom_municipio' a ser normalizada
        nome_correcoes: Dicionário com chaves (nomes incorretos) e valores (nomes corretos)
    
    Returns:
        DataFrame com a coluna 'nom_municipio' normalizada
    """
    # Inicializa a expressão com a coluna original
    expressao = col(coluna)
    
    # Itera sobre o dicionário para construir a lógica de substituição
    for errado, correto in nome_correcoes.items():
        expressao = when(trim(lower(col(coluna))) == errado, correto).otherwise(expressao)
    
    # Aplica a expressão ao DataFrame, substituindo a coluna nom_municipio
    df_normalizado = df.withColumn(coluna, expressao)
    
    return df_normalizado